# Modularity Null Model (Unweighted)

Benchmarks Louvain modularity of the interaction network against 100 degree-preserving randomizations, on the whole network and on its largest connected component (R2 comment 3.2, Table S13).

* **Graph:** undirected, every edge counted once (edge weights ignored).
* **Observed value:** mean modularity of python-louvain runs with seeds 0 to 4.
* **Whole-network null:** `double_edge_swap`, ten swaps per edge.
* **Largest-component null:** `connected_double_edge_swap`, five swaps per edge (connectivity preserved).
* **Seeds:** rewiring seeds drawn from `numpy.random.default_rng(42)`; null run *i* is partitioned with Louvain seed *i*.

Produces the unweighted rows of Table S13. The weighted and Erdős–Rényi rows come from `Modularity Null Model.ipynb`.

**Inputs** (from `derived-data/hashed/`, placed in the working directory): `network_edges_hashed.csv`, `network_nodes_with_community_hashed.csv`.

In [ ]:
!pip -q install python-louvain
import os, sys, csv, time
import multiprocessing as mp
import numpy as np, pandas as pd, networkx as nx
import community as community_louvain   # python-louvain, as used for the paper's partition

PATH_EDGES = 'network_edges_hashed.csv'                 # derived-data/hashed/
PATH_NODES = 'network_nodes_with_community_hashed.csv'  # derived-data/hashed/
R = 100                                                 # number of null randomizations
OUT = 'null_model_unweighted_runs.csv'                  # per-run results, used to resume
print('networkx', nx.__version__)

networkx 3.7


In [ ]:
# Undirected graph with reciprocal edge weights summed (as used for community detection)
nodes = pd.read_csv(PATH_NODES, sep=';')
edges = pd.read_csv(PATH_EDGES)
G = nx.Graph()
G.add_nodes_from(nodes.username)
for s, t, w in edges[['source', 'target', 'weight']].itertuples(index=False):
    if G.has_edge(s, t):
        G[s][t]['weight'] += w
    else:
        G.add_edge(s, t, weight=w)

P = nodes.groupby('community').username.apply(set).tolist()
comps = list(nx.connected_components(G))
print('Paper partition modularity (weighted):', round(nx.community.modularity(G, P, weight='weight'), 4))
print('Connected components:', len(comps), '| two-account components:', sum(len(c) == 2 for c in comps),
      '| largest:', max(map(len, comps)))

# Unweighted copies: whole network and largest connected component
Gu = nx.Graph(G.edges()); Gu.add_nodes_from(G.nodes())
Gg = Gu.subgraph(max(comps, key=len)).copy()

Paper partition modularity (weighted): 0.7991
Connected components: 1616 | two-account components: 1166 | largest: 7137


In [ ]:
def louvain(graph, seed):
    # The unweighted graphs carry no weight attribute, so every edge counts as 1.
    p = community_louvain.best_partition(graph, weight='weight', random_state=seed)
    return community_louvain.modularity(p, graph, weight='weight')

# Rewiring seeds for every run, fixed in advance so parallel and serial runs give identical results
rng = np.random.default_rng(42)
SEEDS = [(int(rng.integers(1e9)), int(rng.integers(1e9))) for _ in range(R)]

def one_run(i):
    s_whole, s_lcc = SEEDS[i]
    h = Gu.copy(); nx.double_edge_swap(h, nswap=10 * h.number_of_edges(), max_tries=10**8, seed=s_whole)
    hg = Gg.copy(); nx.connected_double_edge_swap(hg, nswap=5 * hg.number_of_edges(), seed=s_lcc)
    return i, louvain(h, i), louvain(hg, i)

In [ ]:
# Observed modularity: mean of five Louvain seeds
q_whole = np.mean([louvain(Gu, s) for s in range(5)])
q_lcc = np.mean([louvain(Gg, s) for s in range(5)])
print(f'Observed modularity | whole network {q_whole:.4f} | largest component {q_lcc:.4f}')

Observed modularity | whole network 0.8034 | largest component 0.7361


In [ ]:
# Null randomizations (resumes from OUT if it already holds finished runs)
done = {}
if os.path.exists(OUT):
    with open(OUT) as f:
        for row in csv.DictReader(f):
            done[int(row['run'])] = (float(row['null_whole']), float(row['null_lcc']))
    print(f'Resuming: {len(done)} runs already saved in {OUT}')
todo = [i for i in range(R) if i not in done]

parallel = sys.platform.startswith('linux') and (os.cpu_count() or 1) > 1
print('Mode:', f'parallel on {os.cpu_count()} cores' if parallel else 'serial', '|', len(todo), 'runs to go')
t0 = time.time()
with open(OUT, 'a', newline='') as f:
    w = csv.writer(f)
    if f.tell() == 0:   # write the header only into a new, empty file
        w.writerow(['run', 'null_whole', 'null_lcc'])
    def record(i, a, b):
        done[i] = (a, b); w.writerow([i, a, b]); f.flush()
        if len(done) % 10 == 0 or len(done) == R:
            print(f'  {len(done)}/{R} done ({time.time() - t0:.0f}s)', flush=True)
    if parallel:
        with mp.get_context('fork').Pool(os.cpu_count()) as pool:
            for i, a, b in pool.imap_unordered(one_run, todo):
                record(i, a, b)
    else:
        for i in todo:
            record(*one_run(i))

Mode: parallel on 2 cores | 100 runs to go


  10/100 done (221s)


  20/100 done (443s)


  30/100 done (662s)


  40/100 done (881s)


...

  60/100 done (1323s)


  70/100 done (1548s)


  80/100 done (1768s)


  90/100 done (1990s)


  100/100 done (2204s)


In [ ]:
# Results (Table S13, unweighted rows)
null_whole = np.array([done[i][0] for i in range(R)])
null_lcc = np.array([done[i][1] for i in range(R)])
for label, obs, null in [('Whole network', q_whole, null_whole), ('Largest component', q_lcc, null_lcc)]:
    sd = null.std(ddof=1)
    print(f'{label}: observed {obs:.4f} | null mean {null.mean():.4f} SD {sd:.4f} | '
          f'dQ = {obs - null.mean():.4f} | z = {(obs - null.mean()) / sd:.1f} | '
          f'null runs below observed = {int((null < obs).sum())}/{R}')

Whole network: observed 0.8034 | null mean 0.7314 SD 0.0017 | dQ = 0.0720 | z = 42.4 | null runs below observed = 100/100
Largest component: observed 0.7361 | null mean 0.6526 SD 0.0008 | dQ = 0.0835 | z = 111.2 | null runs below observed = 100/100
